# Ensemble

Ranges of outcomes across an ensemble recipe's members ([ADR-006](../docs/architecture/ADR-006-ensemble-scenarios.md)): pick a recipe generated and run via `sim run <recipe> --ensemble-size N`, see its members' net worth (or any one account's balance) as a percentile-band ("fan") chart, plus a year-end table of mean/median/SD, the 5th-95th percentiles and the tail means -- the same numbers `sim ensemble summary <recipe>` prints. Then compare two recipes head to head on risk, not just their member 0 -- the same numbers `sim ensemble compare <baseline> <variant>` prints -- and a baseline against several recipes in one table.

An ensemble recipe is an ordinary scenario whose `calendar.csv` has one or more rows with `vary_distribution` set; `sim run` detects this and generates `state/runs/<recipe>/member_NNNN.sqlite3` per member instead of a single run file. Recipes with no members yet don't appear in the dropdowns below.

`sim run` also stores each batch's member series in one file beside the members (`series.sqlite3`), so a recipe loads in about a second. A batch run before that, or read by other code since, has its store built the first time it's shown, with a progress bar, and loads quickly after that. Every widget below reads the same stores, so a recipe one has loaded is ready for the others.

> **Note:** after running cells, `git status` will show this notebook as modified even with no real changes. Staging it (`git add`) will make it show unmodified again, unless actual changes are present -- see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it) for why.

In [ ]:
%load_ext autoreload
%autoreload 2

from analysis.sim.notebook_ui.ensemble_widgets import (
    EnsembleControls, ensemble_compare_widget, ensemble_table_widget, ensemble_widget,
)

# The recipe the widgets start on (the browser's recipe, the comparisons'
# baseline); None starts on the first one listed. Each widget picks its own
# recipes after that.
DEFAULT_BASE = "base_ensemble"
# Scale, dollars, account, reserve and the years shown, shared by the
# widgets: a change in one redraws them all. Each kind of widget gets its own
# Lot dropdown, since each lists only the lots of the recipes it shows.
CONTROLS = EnsembleControls()


## Browse an ensemble

- **Recipe**: an ensemble scenario with at least one generated member under `RUNS_DIR`. Re-run the cell below after generating a new one.
- **Account**: net worth, or any one account's balance. **Lot** appears for an account with more than one lot: one lot's balance, or all of them. It lists only the lots of the recipes shown: the picked recipe here, the baseline and picked variants in the comparison.
- While a recipe's series store is built (its first showing, for a batch run before stores), a progress bar under the controls counts its members; the previous chart stays until the new one is ready. The comparisons below do the same.
- The scale, dollars, account and reserve controls are shared with the comparisons below (`CONTROLS`), so changing one redraws them all. The recipes are not: each widget picks its own. `DEFAULT_BASE` names the recipe to start on.
- **Reserve**: CPI-constant dollars. Blank uses the recipe's `scenario.csv` `meta_json` `reserve`, which applies to net worth only; a typed value applies to whatever is shown.
- **Chart**: every month's mean and median across members, over quintile bands shaded darker toward the middle and out to the 5th-95th percentiles, with the reserve as a dashed line in CPI-constant dollars.
- **Table**: mean, median, SD, the `p05`, `p20`-`p80`, `p95` percentiles, and `cvar05`/`cvar20`, the mean of the worst 5% and 20% of members, one row per year-end. With a reserve, `below_reserve` is the share of members below it at that year end and `ever_below_reserve` the share that have been below it at any month so far. No min or max: with many members they're one member's extreme luck.

In [ ]:
ensemble_widget(show_description=True, controls=CONTROLS, default=DEFAULT_BASE)

## Compare two ensembles on risk

A baseline and one other recipe, head to head, for net worth or one account. **Compare** lists only the recipes that pair with the baseline: those sharing its seed (a child inherits its parent's), so member *i* of each drew the same future. Picking the compared recipe as the baseline swaps the two. Charts only: the year-end numbers are in the table below.

- **Percentile deltas**: each breakpoint minus the same breakpoint of the baseline. Δp05 above zero and Δp95 below it means the variant raises the floor and gives up the top. The members at a breakpoint are different futures on each side.
- **Future by future**: the two recipes share a seed, so member *i* of each drew the same future for every row they share, and `variant − baseline` member by member is a real per-future difference. `P(ahead)`, in the table below, is the share of those futures where the variant comes out ahead.
- **CVaR deltas**: the change in the mean of the worst 5% and 20% of members. A fixed, standard pair: steadier than one breakpoint, and it says how bad the bad futures are.
- **Log scale**: the dollar delta panels go on a symmetric log scale, since deltas run both ways; it's linear within 1% of the largest swing.

With few members the low percentiles are noisy; rerun with more members (about 80 for a quick look, 400 to validate) rather than reading small differences.

In [ ]:
ensemble_compare_widget(controls=CONTROLS, default=DEFAULT_BASE)

## Compare several ensembles in one table

A baseline against any number of recipes that pair with it, by year end, each row one (year, recipe):

- **Against the baseline**: each recipe's percentile, CVaR and reserve-share deltas and `P(ahead)`, as the comparison above shows for one.
- **Tail and reserve**: every recipe's tail means and reserve shares, the baseline's first in each year.

Its baseline and recipes are picked here, apart from the comparison above; the scale, dollars, account, reserve and years are the shared ones.

In [ ]:
ensemble_table_widget(controls=CONTROLS, default=DEFAULT_BASE)